In [1]:
# import sys
# !{sys.executable} -m pip install opentrons
#!pip install opentrons

In [45]:
## Imports
from opentrons import protocol_api, execute, simulate #, containers
from opentrons.types import Location, Point
import pandas as pd
import numpy as n
import string
#from data import data_colordf

# Define locations of stock solution

In [26]:
##Defining locations - map
x5 = 118.5 + 13.3    ## x5 & y5 are the coordinates for the bottom left corner of plate 5, same idea for x2 & y2 
y5 = 79.3 + 12.3    
x2 = 118.5 + 13.3    ## x_code = x_irl + 13.3
y2 = -11 + 12.3      ## y_code = y_irl + 12.3 
x4 = 0 + 8.00
y4 = 79.3 + 12.3 

## positions of ice tray set in grid 2
p0 = [[x2 + 0.125*132.4, y2 + 0.75*90.3],#0
      [x2 + 0.375*132.4, y2 + 0.75*90.3],#1
      [x2 + 0.625*132.4, y2 + 0.75*90.3],#2
      [x2 + 0.875*132.4, y2 + 0.75*90.3],#3
      [x2 + 0.125*132.4, y2 + 0.25*90.3],#4
      [x2 + 0.375*132.4, y2 + 0.25*90.3],#5
      [x2 + 0.625*132.4, y2 + 0.25*90.3],#6
      [x2 + 0.875*132.4, y2 + 0.25*90.3]]#7

#FOR WHEN STOCK IS IN 5
# p1 = [x5+0.25*132.4, y5 + 0.75*90.3]
# p2 = [x5 + 0.75*132.4,p1[1]]
# p3 = [p1[0],y5 + 0.25*90.3]
# p4 = [p2[0],p3[1]]

p1 = [x4+0.25*132.4, y4 + 0.75*90.3] # water
p2 = [x4 + 0.75*132.4-10,p1[1]] # blue 
p3 = [p1[0],y4 + 0.25*90.3] # Red
p4 = [p2[0],p3[1]] # yellow

In [27]:
## The coordinates are absolute 
## with reference to the bottom left corner of slot 1 as origin.
## x, y, and z can be a float or integer

loc0 = [
    Location(Point(p0[0][0],p0[0][1],6),None),
    Location(Point(p0[1][0],p0[1][1],6),None),
    Location(Point(p0[2][0],p0[2][1],6),None),
    Location(Point(p0[3][0],p0[3][1],6),None),
    Location(Point(p0[4][0],p0[4][1],6),None),
    Location(Point(p0[5][0],p0[5][1],6),None),
    Location(Point(p0[6][0],p0[6][1],6),None),
    Location(Point(p0[7][0],p0[7][1],6),None)
]

loc1 = Location(Point(p1[0],p1[1],6),None) # location of water
loc2 = Location(Point(p2[0],p2[1],6),None) # location of blue dye 
loc3 = Location(Point(p3[0],p3[1],6),None) # location of red dye
loc4 = Location(Point(p4[0],p4[1],6),None)  # location of yellow dye

# Call protocol to simulate or Execute

In [28]:
protocol = simulate.get_protocol_api('2.10')
#protocol = execute.get_protocol_api('2.15')

/Users/clarat/.opentrons/robot_settings.json not found. Loading defaults
Deck calibration not found.
/Users/clarat/.opentrons/deck_calibration.json not found. Loading defaults


# Define labwear

In [29]:
## pipette and labware
tiprack = protocol.load_labware('opentrons_96_tiprack_300ul', '8')
pip = protocol.load_instrument('p300_single_gen2', 'right', tip_racks=[tiprack])
plate = protocol.load_labware('opentrons_96_wellplate_200ul_pcr_full_skirt','2')
#flat_plate = protocol.load_labware('nest_96_wellplate_200ul_flat','2')

# Relevant Functions

In [30]:
# funtion for moving liquid since repetative
def pipette_action(protocol, pipette, stock_volume_to_pull, stock_position_to_pull, destination_well,stockname):
    pipette.aspirate(stock_volume_to_pull, stock_position_to_pull)
    pipette.dispense(stock_volume_to_pull, destination_well)
    pipette.touch_tip()  
    #pipette.blow_out(destination_well)

In [31]:
def stock_in_all(stock,stock_location,tip_max_vol,dis_tray,name):
    '''
    Args- 
    stock: 1D array of all the stock volumes
    stock_location: 1D array of the stock locations corresponding to volumes
    tip_max_vol: float num of the max volume tip can hold
    dis_tray: 1D array of all the locaitons for dispensing
    name: stock name str

    Function-
    Determine the number of times needed to get stock and dispence. 
    If the volume is larger than the tip can hold, automatically determined how many times it has to aspirate and dispence. 
    Care since stock, stock_location, dis_tray are all arrays
    '''
    for i in range(len(stock)):
        stock_amount = stock[i]
        if stock_amount == 0:
            pass
        elif stock[i] >= tip_max_vol: # eg:300 is max tip volume
            n = stock[i]//tip_max_vol
            if n > 0: 
                for count in range(int(n)):
                    pipette_action(protocol, 
                            pipette = pip, 
                            stock_volume_to_pull = tip_max_vol, 
                            stock_position_to_pull = stock_location[i], 
                            destination_well = dis_tray[i],
                            stockname= name )
                    stock_amount -= tip_max_vol
                    
                    if stock_amount <= stock[i] - tip_max_vol*n and stock_amount != 0:
                        pipette_action(protocol, 
                                    pipette = pip, 
                                    stock_volume_to_pull = (stock[i] - tip_max_vol*n),
                                    stock_position_to_pull = stock_location[i], 
                                    destination_well = dis_tray[i],
                                    stockname= name )
        else: 
            pipette_action(protocol, 
                           pipette = pip, 
                           stock_volume_to_pull = stock[i], 
                           stock_position_to_pull = stock_location[i], 
                           destination_well = dis_tray[i],
                           stockname= name )
        

<bound method Labware.wells of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2>


In [46]:
def loc_stock(n_samples,fill):
    array = []
    #fill array
    for i in range(n_samples): 
        array.append(fill) 
    return array


def loc_mixing(start_let,start_n,n_sample): 
    letters = string.ascii_uppercase[0:8]
    let_id = letters.index(start_let)

    array = []
    for n in range(1,n_sample):
        array.append(plate.wells()[start_n+let_id*n+12])
        array.append(plate.rows()[start_n+let_id*n+12])
        
    return array

#let = 'D'
#RGB_stocksandlocations = {'mix_loc': [plate[str(let)+'1'].center(), plate[str(let)+'2'].center(),plate[str(let)+'3'].center(),plate[str(let)+'4'].center(),plate[str(let)+'5'].center(),plate[str(let)+'6'].center(),plate[str(let)+'7'].center(),plate[str(let)+'8'].center(),plate[str(let)+'9'].center(),plate[str(let)+'10'].center(),plate[str(let)+'11'].center(),plate[str(let)+'12'].center()], 

https://docs.opentrons.com/ot1/containers.html

# Volumes and locations

In [11]:
##Calling the function
N_samples = 36

loc_red_stoc = loc_stock(N_samples,loc2)
loc_blue_stoc = loc_stock(N_samples,loc3)
loc_yellow_stoc = loc_stock(N_samples,loc4)

loc_mixing = loc_mixing(0,N_samples) #Start is 0 - 95, A: 0-11, B: 12-23, 

RGB_stocksandlocations = {'mix_loc': loc_mixing,

## Clara BO iteration: 
'stock_red_vol': [135., 156., 147., 136., 145., 149.,   7.,  11.,  24., 127., 162., 41., 115., 146., 140., 202., 112.,   0.,  95., 100.,  94., 300., 111., 105.,   0.,   0., 300., 109., 108., 108., 107., 110., 107., 107., 109., 107.],
'stock_blue_vol':[ 31., 117., 106.,  11.,   0., 148., 166., 233., 169.,  60., 137., 67., 184., 153., 159.,   0., 187., 300., 178., 100., 205.,   0.,188., 194., 150.,   0.,   0., 190., 191., 191., 192., 189., 192.,192., 190., 192.],
'stock_yellow_vol':[132.,  26.,  45., 151., 154.,   1., 126.,  55., 105., 111.,   0.,190.,   0.,   0.,   0.,  97.,   0.,   0.,  26., 100.,   0.,   0., 0.,   0., 150., 300.,   0.,   0.,   0.,   0.,   0.,   0.,   0., 0.,   0.,   0.],

## Clara LHS 2/23/2024                          
    # 'stock_red_vol':         [ 82., 165., 105.,  26., 109.,  84.,  32., 214., 114., 139., 214., 15.],
    # 'stock_blue_vol':        [ 39.,  76.,  78., 122., 128., 124., 153.,  37., 125., 100.,  39., 69.],
    # 'stock_yellow_vol':      [177.,  57., 116., 150.,  62.,  91., 114.,  47.,  59.,  59.,  46., 214.] ,                          

## Arthur 2/16/2024
#     'stock_red_vol':         [ 100,   50,   200,  50,  150,  0 ,  150,  25,  75,    25,  75,   100],
#     'stock_blue_vol':        [ 100,  200,  50,   50,  150 , 150 ,  0 ,  75,  25,   175,  200,  70],
#     'stock_yellow_vol':      [ 100,   50,  50,   200,   0 ,  150, 150,  200, 200,  100,  25,   130],
#     'stock_black_vol':       [   0,    0,    0,    0,    0,   0,   0,    0,   0,    0,    0,    0],
                          
## Arthur Rainbow
                           ## red    ora  yel   gre   blu   ind   vio    all 
#    'stock_red_vol':         [85,  95,   0,  0,   0, 95, 152,  50,  0,  0,  0,  0],
#    'stock_blue_vol':        [  0,   0,   0, 95, 180, 95,  38, 50,  0,  0,  0,  0],
#    'stock_yellow_vol':      [ 35,  95, 180, 95,   0,  0,   0, 50,  0,  0,  0,  0],
#    'stock_black_vol':       [ 70,   0,  10,  0,  10,  0,   0, 40,  0,  0,  0,  0],
                          
#     'stock_red_vol':         [170, 150, 150, 140, 130,  0,   0,  0],
#     'stock_blue_vol':        [ 30,   0,   0,   0,  30,  0,   0,  0],
#     'stock_yellow_vol':      [  0,   0,  40,  40,  30,  0,   0,  0],
#     'stock_black_vol':       [  0,  50,   0,  10,   0,  0,   0,  0],
                          
    'loc_red_stock':         loc_red_stoc,
    'loc_blue_stock':        loc_blue_stoc,
    'loc_yellow_stock':      loc_yellow_stoc,
    }
RGB_stocks = pd.DataFrame(data=RGB_stocksandlocations)
RGB_stocks.head()

,mix_loc,stock_red_vol,stock_blue_vol,stock_yellow_vol,loc_red_stock,loc_blue_stock,loc_yellow_stock
0,A1 of Opentrons Tough 96 Well Plate 200 µL PCR...,135.0,31.0,132.0,"Location(point=Point(x=97.30000000000001, y=15...","Location(point=Point(x=41.1, y=114.175, z=6), ...","Location(point=Point(x=97.30000000000001, y=11..."
1,B1 of Opentrons Tough 96 Well Plate 200 µL PCR...,156.0,117.0,26.0,"Location(point=Point(x=97.30000000000001, y=15...","Location(point=Point(x=41.1, y=114.175, z=6), ...","Location(point=Point(x=97.30000000000001, y=11..."
2,C1 of Opentrons Tough 96 Well Plate 200 µL PCR...,147.0,106.0,45.0,"Location(point=Point(x=97.30000000000001, y=15...","Location(point=Point(x=41.1, y=114.175, z=6), ...","Location(point=Point(x=97.30000000000001, y=11..."
3,D1 of Opentrons Tough 96 Well Plate 200 µL PCR...,136.0,11.0,151.0,"Location(point=Point(x=97.30000000000001, y=15...","Location(point=Point(x=41.1, y=114.175, z=6), ...","Location(point=Point(x=97.30000000000001, y=11..."
4,E1 of Opentrons Tough 96 Well Plate 200 µL PCR...,145.0,0.0,154.0,"Location(point=Point(x=97.30000000000001, y=15...","Location(point=Point(x=41.1, y=114.175, z=6), ...","Location(point=Point(x=97.30000000000001, y=11..."


In [12]:
blue_stock_vol   = pd.Series.to_list(RGB_stocks['stock_blue_vol'])
red_stock_vol    = pd.Series.to_list(RGB_stocks['stock_red_vol'])
yellow_stock_vol = pd.Series.to_list(RGB_stocks['stock_yellow_vol'])

blue_stock_loc   = pd.Series.to_list(RGB_stocks['loc_blue_stock'])
red_stock_loc    = pd.Series.to_list(RGB_stocks['loc_red_stock'])
yellow_stock_loc = pd.Series.to_list(RGB_stocks['loc_yellow_stock'])

dispense =pd.Series.to_list(RGB_stocks['mix_loc'])#np.array(RGB_stocks['mix_loc'])

# Run the protocol

In [13]:
## commands
#protocol.home()
# move red dye

pip.transfer(red_stock_vol,loc2,dispense,blow_out=True,blowout_location="destination well" ,new_tip='once')
#pip.drop_tip()
# move blue 
pip.transfer(blue_stock_vol,loc3,dispense,blow_out=True,blowout_location="destination well" ,new_tip='once') 
# #pip.drop_tip()
# # move yellow
pip.transfer(yellow_stock_vol,loc4,dispense,blow_out=True,blowout_location="destination well",mix_after=(5, 75),new_tip='once') 
   
for line in protocol.commands(): 
    print(line)

When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over aspiration if the previous command is a blow_out.
When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over aspiration if the previous command is a blow_out.
When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over aspiration if the previous command is a blow_out.
When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over aspiration if the previous command is a blow_out.
When aspirate is called on something other than a well relative position, we can't move to the top of the well to prepare for aspiration. This might cause over 

Transferring [135.0, 156.0, 147.0, 136.0, 145.0, 149.0, 7.0, 11.0, 24.0, 127.0, 162.0, 41.0, 115.0, 146.0, 140.0, 202.0, 112.0, 0.0, 95.0, 100.0, 94.0, 300.0, 111.0, 105.0, 0.0, 0.0, 300.0, 109.0, 108.0, 108.0, 107.0, 110.0, 107.0, 107.0, 109.0, 107.0] from (97.30000000000001, 159.325, 6) to A1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2
Picking up tip from A1 of Opentrons OT-2 96 Tip Rack 300 µL on 8
Aspirating 135.0 uL from (97.30000000000001, 159.325, 6) at 92.86 uL/sec
Dispensing 135.0 uL into A1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2 at 92.86 uL/sec
Blowing out at A1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2
Aspirating 156.0 uL from (97.30000000000001, 159.325, 6) at 92.86 uL/sec
Dispensing 156.0 uL into B1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2 at 92.86 uL/sec
Blowing out at B1 of Opentrons Tough 96 Well Plate 200 µL PCR Full Skirt on 2
Aspirating 147.0 uL from (97.30000000000001, 159.325, 6) at 92.86 uL/